# MPPT Algorithm Comparison: Results Overview

Interactive exploration of the Monte Carlo simulation results comparing the
5 MPPT algorithms in this paper -- spanning perturbative (P&O, IncCond),
rule-based (Fuzzy Logic), learning-based (Q-learning), and model-based
(Sliding Mode Control) paradigms -- across 11 test scenarios (steady state,
irradiance/temperature transients, partial shading, sensor noise, and
rapid fluctuation).

**Key metrics:** tracking efficiency (%), convergence time (s), settling
time (s), steady-state oscillation, energy yield ratio, and per-step
execution time. See `CLAUDE.md` for full scenario and metric definitions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('../styles/publication.mplstyle')
%matplotlib inline

In [ ]:
# Per (algorithm, scenario, metric) descriptive statistics, produced by:
#   python -m src.analysis --input results/comparison_table.csv --output results/
summary = pd.read_csv('../results/summary_table.csv')
print(f"Rows: {len(summary)}")
print(f"Algorithms: {sorted(summary['algorithm'].unique())}")
print(f"Scenarios: {sorted(summary['scenario'].unique())}")
summary.head()

In [ ]:
# Tracking efficiency heatmap: algorithm x scenario
eff = summary[summary['metric'] == 'tracking_efficiency_pct']
pivot_eff = eff.pivot_table(values='mean', index='scenario', columns='algorithm')

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(pivot_eff.values, cmap='viridis', aspect='auto')
ax.set_xticks(range(len(pivot_eff.columns)))
ax.set_xticklabels(pivot_eff.columns, rotation=45, ha='right')
ax.set_yticks(range(len(pivot_eff.index)))
ax.set_yticklabels(pivot_eff.index)
for i in range(pivot_eff.shape[0]):
    for j in range(pivot_eff.shape[1]):
        val = pivot_eff.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f"{val:.1f}", ha='center', va='center', color='white', fontsize=8)
ax.set_title('Tracking Efficiency (%) by Algorithm and Scenario')
fig.colorbar(im, ax=ax, label='Tracking efficiency (%)')
plt.tight_layout()
plt.savefig('../results/figures/notebook_efficiency_heatmap.png', dpi=300)
plt.show()

In [ ]:
# Convergence time distribution per algorithm, from the raw Monte Carlo runs
raw = pd.read_csv('../results/comparison_table.csv')
conv = raw[raw['metric'] == 'convergence_time_s'].dropna(subset=['value'])

fig, ax = plt.subplots(figsize=(12, 6))
algorithms = sorted(conv['algorithm'].unique())
data = [conv[conv['algorithm'] == a]['value'].values for a in algorithms]
ax.boxplot(data, tick_labels=algorithms)
ax.set_xticklabels(algorithms, rotation=45, ha='right')
ax.set_ylabel('Convergence time (s)')
ax.set_title('Convergence Time Distribution Across Scenarios')
plt.tight_layout()
plt.savefig('../results/figures/notebook_convergence_boxplot.png', dpi=300)
plt.show()